# 🎙️ Audio Transcription v2 — faster-whisper + Speaker Diarization

**Runtime:** Google Colab — T4 GPU  
**What it does:** Transcribes audio and labels each segment by speaker (SPEAKER_00, SPEAKER_01, etc.)

---
### Before running:
1. `Runtime → Change runtime type → T4 GPU`
2. Upload your audio file to Google Drive
3. Get a Hugging Face token and accept gated model terms (see Cell 4)
4. Update `AUDIO_FILE` and `HF_TOKEN` in Cell 5

## 1. Install

In [ ]:
!pip install -q faster-whisper pyannote.audio

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/drive')
print("✅ Drive mounted")

## 3. Convert audio to WAV
pyannote requires clean WAV format — this avoids decoder errors with mp3 files.

In [ ]:
import os

AUDIO_FILE = "/drive/MyDrive/your_audio_file.mp3"  # ← update
WAV_FILE   = "/content/audio.wav"

!ffmpeg -err_detect ignore_err -i "{AUDIO_FILE}" -ar 16000 -ac 1 -c:a pcm_s16le "{WAV_FILE}" -y -loglevel error

size = os.path.getsize(WAV_FILE) / 1024**2
assert size > 1, "WAV file too small — conversion likely failed"
print(f"✅ Converted: {size:.1f} MB")

## 4. Hugging Face setup

One-time steps (do these before running Cell 5):

1. Sign up at [huggingface.co](https://huggingface.co)
2. Accept terms at **both** of these URLs:
   - [huggingface.co/pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1)
   - [huggingface.co/pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0)
3. Go to **Settings → Access Tokens → New token** (read access) → copy it

## 5. Transcribe (Whisper)

In [ ]:
import time, warnings
from faster_whisper import WhisperModel

warnings.filterwarnings("ignore")

MODEL_SIZE = "base"   # tiny | base | small | medium | large-v3
LANGUAGE   = None     # e.g. "en" or None to auto-detect

print(f"Loading Whisper '{MODEL_SIZE}'...")
whisper_model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")

print("Transcribing...")
t0 = time.time()
options = {"language": LANGUAGE} if LANGUAGE else {}
segments, info = whisper_model.transcribe(WAV_FILE, **options)
segments = list(segments)

print(f"✅ Done in {time.time()-t0:.1f}s — language: {info.language}, segments: {len(segments)}")

## 6. Diarise (pyannote)

In [ ]:
import torch
from pyannote.audio import Pipeline

HF_TOKEN = "your_token_here"  # ← paste your Hugging Face token

print("Loading diarization pipeline...")
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN
)
pipeline.to(torch.device("cuda"))

print("Running diarization...")
t0 = time.time()
diarization = pipeline(WAV_FILE)
print(f"✅ Done in {time.time()-t0:.1f}s")

## 7. Merge transcript + speakers

In [ ]:
def fmt(s):
    return f"{int(s//60):02d}:{int(s%60):02d}"

def get_speaker(start, end, diarization):
    overlap = {}
    for turn, _, speaker in diarization.speaker_diarization.itertracks(yield_label=True):
        o = min(turn.end, end) - max(turn.start, start)
        if o > 0:
            overlap[speaker] = overlap.get(speaker, 0) + o
    return max(overlap, key=overlap.get) if overlap else "UNKNOWN"

results = []
for seg in segments:
    speaker = get_speaker(seg.start, seg.end, diarization)
    line = f"[{fmt(seg.start)} → {fmt(seg.end)}]  {speaker}: {seg.text.strip()}"
    results.append(line)
    print(line)

## 8. Save & download

In [ ]:
from google.colab import files

out_path = "/content/transcript_speakers.txt"

with open(out_path, "w", encoding="utf-8") as f:
    f.write("\n".join(results))

print("✅ Saved")
files.download(out_path)